# 02. Corpus Exploration — スナップショットの解剖

## 学習目標
- 再現可能なコーパススナップショット（smoke/pilot/main/validation/calibration/test）の設計を理解する
- 文書長分布・文字種構成・日本語率・頻出文字を実測し、Web（FineWeb2）と Wikipedia の差を見る
- train/validation/calibration/test の分離がなぜ本質的かを説明できる

## スナップショット設計（spec §4.1）
1つの決定論的ストリーム走査で、フィルタ後の文書 index `i` に対し
`i%50==0→validation, ==1→calibration, ==2→test, それ以外→train pool` と割り当てる。
smoke⊂pilot⊂main は train pool の**先頭プレフィックス**なので、小実験は大実験の始点そのもの。
validation/calibration/test は**別文書**なので全学習集合と交わらない（暗記と汎化を分離できる）。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
manifest = load_json(ROOT / "data/manifests/snapshots_v1.json")
stats = load_json(ROOT / "reports/figures/corpus_stats.json")

rows = []
for name, snap in manifest["snapshots"].items():
    if "n_chars" in snap:
        rows.append((name, snap.get("source",""), snap.get("n_docs","-"), snap["n_chars"]))
df = pd.DataFrame(rows, columns=["snapshot","source","n_docs","n_chars"]).sort_values("n_chars")
print("フィルタ設定:", manifest["filter"])
print("FineWeb2ja: streamed", manifest["sources"]["fineweb2ja"]["streamed_docs"],
      "kept", manifest["sources"]["fineweb2ja"]["kept_docs"],
      "filtered", manifest["sources"]["fineweb2ja"]["filtered_out"])
df

フィルタ設定: {'min_chars': 200, 'max_chars': 100000, 'min_japanese_ratio': 0.5}
FineWeb2ja: streamed 122552 kept 117581 filtered 4971


,snapshot,source,n_docs,n_chars
2,calibration,fineweb2ja,1033,1700606
1,validation,fineweb2ja,990,1700677
3,test,fineweb2ja,1113,1701266
0,wiki_pilot,wikipedia_ja,1949,17002682
4,train_pool,fineweb2ja,114445,170005924


In [3]:
# 文書長分布: FineWeb2 (pilot) vs Wikipedia
fig = go.Figure()
for name, color in [("pilot","#1f77b4"), ("wiki_pilot","#ff7f0e")]:
    lens = stats["snapshots"][name]["doc_len_hist"]
    fig.add_trace(go.Histogram(x=lens, name=name, opacity=0.6, marker_color=color, nbinsx=50))
fig.update_layout(barmode="overlay", title="文書長分布（文字数）: Web vs Wikipedia",
                  xaxis_title="文書の文字数", yaxis_title="文書数（サンプル）",
                  template="plotly_white", height=380)
fig.update_xaxes(range=[0, 6000])
fig.show()
for name in ["pilot","wiki_pilot"]:
    d = stats["snapshots"][name]["doc_len"]
    print(f'{name:11s} median {d["median"]:>6} chars  mean {d["mean"]:>7}  max {d["max"]:,}')

pilot       median    808 chars  mean  1572.8  max 59,580
wiki_pilot  median   4969 chars  mean  8723.8  max 96,325


In [4]:
# 文字種構成: Web vs Wikipedia
classes = ["hiragana","katakana","kanji","ascii_alnum","jp_punct","other","whitespace"]
fig = go.Figure()
for name, color in [("pilot","#1f77b4"),("wiki_pilot","#ff7f0e")]:
    frac = stats["snapshots"][name]["class_fraction"]
    fig.add_trace(go.Bar(x=classes, y=[frac.get(c,0) for c in classes], name=name, marker_color=color))
fig.update_layout(barmode="group", title="文字種構成: FineWeb2(pilot) vs Wikipedia",
                  yaxis_title="割合", template="plotly_white", height=380)
fig.show()
for name in ["pilot","wiki_pilot"]:
    print(f'{name:11s} 日本語率 {stats["snapshots"][name]["japanese_ratio_mean"]:.3f}')

pilot       日本語率 0.886
wiki_pilot  日本語率 0.847


In [5]:
# 頻出文字 top20（pilot）
top = stats["snapshots"]["pilot"]["top_chars"][:20]
fig = go.Figure(go.Bar(x=[t["char"] for t in top], y=[t["count"] for t in top], marker_color="#1f77b4"))
fig.update_layout(title="FineWeb2(pilot) 頻出文字 top20", xaxis_title="文字", yaxis_title="出現数",
                  template="plotly_white", height=340)
fig.show()

In [6]:
# token 単位の情報（chars/token = 圧縮率）
tk = manifest["tokenized"]
rows = [(k.replace("_bpe8k_v1",""), v["n_tokens"], v["chars_per_token"]) for k,v in tk.items()]
pd.DataFrame(rows, columns=["snapshot","n_tokens","chars_per_token"]).sort_values("n_tokens")

,snapshot,n_tokens,chars_per_token
0,smoke,1054722,1.6144
4,test,1079368,1.5778
2,validation,1079739,1.5765
3,calibration,1092315,1.5584
1,pilot,10523324,1.6171
5,wiki_pilot,11737426,1.4488
6,main,108465345,1.5690


## Observation / Interpretation / Caveat
- **Observation**: Web(FineWeb2) は Wikipedia より文書長のばらつきが大きく、ASCII 比率が高い。日本語率は両者とも高い（フィルタ 0.5 以上）。BPE8K の圧縮率は約 1.6 文字/トークン。
- **Interpretation**: Web はノイズ（広告・定型文）と多様性を含み、Wikipedia は均質で百科事典的。学習コーパスの性格はモデルの生成傾向に直接効く（M4 で比較）。
- **Caveat**: 統計は各スナップショットの先頭数千文書のサンプル。フィルタ（min_japanese_ratio=0.5, 200–100k字）が分布を既に整形している。exact dedup は行ったが near-dup は未処理。

## 確認問題
1. なぜ validation を train の末尾から取るのではなく、別文書にすべきか。
2. smoke が pilot の先頭プレフィックスであることの再現性上の利点は。
3. 圧縮率 1.6 文字/トークンは、context 512 が「約何文字」に相当することを意味するか。

**次へ**: [07_architecture_visualization](07_architecture_visualization.ipynb)